In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

def tokenize_func (sentence):
    return tokenizer(sentence['text'], padding = "max_length", truncation=True, return_tensors="pt", max_length=128)

print(model.config.id2label)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{0: 'NEGATIVE', 1: 'POSITIVE'}


In [2]:
from datasets import load_dataset
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
ds

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

c:\Users\Sachio\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sachio\.cache\huggingface\hub\datasets--Salesforce--wikitext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

In [ ]:
tokenized_ds = ds.map(tokenize_func, batched=True)

In [ ]:
print(tokenized_ds)

DatasetDict({
    test: Dataset({
        features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3760
    })
})


In [ ]:
tokenized_ds = tokenized_ds.remove_columns(['text'])


In [ ]:
tokenized_ds = tokenized_ds.with_format("torch")
tokenized_ds

DatasetDict({
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3760
    })
})

In [ ]:
from torch.utils.data import DataLoader

train_ds = DataLoader(tokenized_ds['train'], batch_size=16, shuffle=True)

for step, batch in enumerate(train_ds):
    print(batch['input_ids'].shape)
    if step > 5:
        break


torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])


In [ ]:
import torch
from transformers import AutoTokenizer, DataCollatorWithPadding
from datasets import load_dataset
from torch.utils.data import DataLoader

ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer2 = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_func_new (sentence):
    return tokenizer2(sentence['text'], truncation=True)

tokenized_ds2 = ds.map(tokenize_func_new, batched=True)
tokenized_ds2 = tokenized_ds2.remove_columns(['text'])
tokenized_ds2 = tokenized_ds2.with_format("torch")

data_collator = DataCollatorWithPadding(tokenizer2)
train_dataloader = DataLoader(tokenized_ds2['train'], batch_size=16, shuffle=True, collate_fn=data_collator)

for step, batch in enumerate(train_dataloader):
    print(batch['input_ids'].shape)
    if step > 5:
        break

torch.Size([16, 347])
torch.Size([16, 186])
torch.Size([16, 300])
torch.Size([16, 297])
torch.Size([16, 351])
torch.Size([16, 484])
torch.Size([16, 307])


In [ ]:
from datasets import load_dataset

ds_glue = load_dataset("nyu-mll/glue", "cola")

{'sentence': ["Our friends won't buy this analysis, let alone the next one we propose.",
  "One more pseudo generalization and I'm giving up.",
  "One more pseudo generalization or I'm giving up.",
  'The more we study verbs, the crazier they get.',
  'Day by day the facts are getting murkier.'],
 'label': [1, 1, 1, 1, 1],
 'idx': [0, 1, 2, 3, 4]}

In [ ]:
import torch
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

ds_glue = load_dataset("nyu-mll/glue", "cola")
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer2 = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

def tokenize_func_new (sentences):
    return tokenizer2(sentences['sentence'], truncation=True)

tokenized_ds2 = ds_glue.map(tokenize_func_new, batched=True)

data_collator = DataCollatorWithPadding(tokenizer2)
training_args = TrainingArguments(
    "test-trainer", 
    num_train_epochs=5, 
    learning_rate=2e-5, 
    weight_decay=0.01, 
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    eval_strategy="epoch",   
    gradient_accumulation_steps=4,
    lr_scheduler_type="cosine",
    fp16=True
    #still many hyperparemeters
)

def compute_metrics(eval_preds):
    # 1. Load the accuracy metric
    metric = evaluate.load("glue", "cola")
    
    # 2. Unpack the predictions and the true labels
    logits, labels = eval_preds
    
    # 3. Convert raw model logits into class IDs (0 or 1)
    preds = np.argmax(logits, axis=-1)
    
    # 4. Compute and return the results as a dictionary
    return metric.compute(predictions=preds, references=labels)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_ds2["train"],
    eval_dataset=tokenized_ds2["validation"],
    data_collator=data_collator,
    processing_class=tokenizer2,
    compute_metrics= compute_metrics,
)

trainer.train()


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Matthews Correlation
1,No log,0.515513,0.356453
2,No log,0.485889,0.426621
3,No log,0.523402,0.458734
4,1.784129,0.551524,0.469735
5,1.784129,0.549046,0.468431


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=670, training_loss=1.558604681669776, metrics={'train_runtime': 117.6834, 'train_samples_per_second': 363.305, 'train_steps_per_second': 5.693, 'total_flos': 229000686898068.0, 'train_loss': 1.558604681669776, 'epoch': 5.0})

In [ ]:
prediction = trainer.predict(tokenized_ds2["validation"])
print(prediction.predictions.shape, prediction.label_ids.shape)


(1043, 2) (1043,)


In [ ]:
import numpy as np
import evaluate

# 1. Extract the raw logit arrays from the prediction output
logits = prediction.predictions

# 2. Take the argmax along the last axis to get predicted class IDs (0 or 1)
preds = np.argmax(logits, axis=-1)

# 3. Load the evaluate metric
metric = evaluate.load("accuracy")

# 4. Compute the score using predictions and ground-truth references
results = metric.compute(predictions=preds, references=tokenized_ds2["validation"]["label"])
print(results)

{'accuracy': 0.8063279002876318}


In [5]:
%pip install --upgrade datasets torchvision transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 113.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 14.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 5.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 91.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
%pip install accelerate

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_scheduler
)
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
import evaluate
from accelerate import Accelerator
import sys
sys.modules.pop("torchvision", None)
# -----------------------
# Dataset
# -----------------------

# loads the data set 
raw_datasets = load_dataset("nyu-mll/glue", "cola")
checkpoint = "distilbert-base-uncased"
tokenizer3 = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_fn (phrases):
    return tokenizer3(phrases['sentence'], truncation=True)

tokenized_data = raw_datasets.map(tokenize_fn, batched=True)
tokenized_data = tokenized_data.remove_columns(['sentence', 'idx'])
tokenized_data = tokenized_data.rename_column('label', 'labels')
tokenized_data.set_format("torch")
Data_Collator = DataCollatorWithPadding(tokenizer=tokenizer3)

DataTrainLoader = DataLoader(tokenized_data['train'], shuffle=True, batch_size=8, collate_fn=Data_Collator)
DataEvalLoader = DataLoader(tokenized_data['validation'], batch_size=8, collate_fn=Data_Collator)

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

Optimizer = AdamW(model.parameters(), lr=2e-5)

accelerator = Accelerator()
model, Optimizer, DataTrainLoader, DataEvalLoader = accelerator.prepare(model, Optimizer, DataTrainLoader, DataEvalLoader)

epochs = 3

num_training_steps = epochs * len(DataTrainLoader)

lr_optimizer = get_scheduler(
    name="linear",
    optimizer=Optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

progress_bar = tqdm(range(num_training_steps))

for epoch in range(epochs):

    model.train()

    metric_train = evaluate.load("glue", "cola")

    running_loss = 0

    for batch in DataTrainLoader:

        Optimizer.zero_grad()

        outputs = model(**batch)

        loss = outputs.loss

        accelerator.backward(loss)

        Optimizer.step()

        lr_optimizer.step()

        running_loss += loss.item()

        preds = torch.argmax(
            outputs.logits,
            dim=-1
        )

        metric_train.add_batch(
            predictions=accelerator.gather(preds),
            references=accelerator.gather(batch["labels"])
        )

    train_metrics = metric_train.compute()

    print(
        f"Epoch {epoch+1} | "
        f"Loss: {running_loss/len(DataTrainLoader):.4f} | "
        f"Metrics: {train_metrics}"
    )


unwrapped_model = accelerator.unwrap_model(model)

unwrapped_model.save_pretrained(
    "./my_bert_model"
)

tokenizer3.save_pretrained("./my_bert_tokenizer")



Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

  0%|          | 0/3207 [00:00<?, ?it/s]

Epoch 1 | Loss: 0.5687 | Metrics: {'matthews_correlation': np.float64(0.23536268713446049)}
Epoch 2 | Loss: 0.3540 | Metrics: {'matthews_correlation': np.float64(0.621151182595149)}
Epoch 3 | Loss: 0.1879 | Metrics: {'matthews_correlation': np.float64(0.829995698058496)}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_bert_tokenizer/tokenizer_config.json',
 './my_bert_tokenizer/tokenizer.json')

In [ ]:
metric = evaluate.load("glue", "cola")
model.eval()
for batches in DataEvalLoader:

    with torch.no_grad():
        eval_output = model(**batches)

    logits = eval_output.logits
    pred = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=accelerator.gather(pred), references=accelerator.gather(batches["labels"]))
results = metric.compute()
print(results)

{'matthews_correlation': np.float64(0.5069898363255262)}


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 8551
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1043
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1063
    })
})